In [ ]:
import os
import zipfile
import tensorflow as tf

URL = "https://download.mlcc.google.com/mledu-datasets/cats_and_dogs_filtered.zip"
DATA_DIR = "/content/cats_and_dogs_filtered"

# Download the zip into /content (file name only, no path)
zip_path = tf.keras.utils.get_file(
    fname="cats_and_dogs_filtered.zip",
    origin=URL,
    cache_dir="/content",
    cache_subdir="",
    extract=False,
)

# Extract it
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall("/content")

train_dir = os.path.join(DATA_DIR, "train")
test_dir = os.path.join(DATA_DIR, "validation")

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 123

# Split training images into training and validation sets
train_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="training",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="binary",
)

# Load the separate test images
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode="binary",
)

class_names = train_ds.class_names

print("Classes:", class_names)
print("Training batches:", tf.data.experimental.cardinality(train_ds).numpy())
print("Validation batches:", tf.data.experimental.cardinality(val_ds).numpy())
print("Test batches:", tf.data.experimental.cardinality(test_ds).numpy())


In [ ]:
import matplotlib.pyplot as plt

# Preview 12 training images and their labels
images, labels = next(iter(train_ds))

plt.figure(figsize=(12, 8))

for i in range(12):
    plt.subplot(3, 4, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(class_names[int(labels[i].numpy()[0])])
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
# Load MobileNetV2 with pretrained ImageNet weights
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights="imagenet"
)

# Keep the pretrained weights unchanged
base_model.trainable = False

print("Total base-model layers:", len(base_model.layers))
print("Base model trainable:", base_model.trainable)

In [ ]:
# Step 2: Data augmentation - random changes to training images
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
], name="data_augmentation")

# Build the classifier on top of MobileNetV2
inputs = tf.keras.Input(shape=(224, 224, 3))

# Augment the raw images before preprocessing
x = data_augmentation(inputs)

# Prepare images for MobileNetV2
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

# Extract features using the frozen base model
x = base_model(x, training=False)

# Add the classification head
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dense(128, activation="relu")(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(1, activation="sigmoid")(x)

model = tf.keras.Model(inputs, outputs)

model.summary()

In [ ]:

# Configure how the model learns
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)


In [ ]:


# Stop training if validation loss stops improving
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True,
    start_from_epoch=9
)

# Train the classification head
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=[early_stopping]
)


In [ ]:
# Phase 1 result: validation accuracy with the base model frozen
phase1_val_loss, phase1_val_accuracy = model.evaluate(val_ds, verbose=0)

print(f"Phase 1 validation accuracy: {phase1_val_accuracy:.2%}")

In [ ]:
# Step 5: Phase 2 fine-tuning
# Unfreeze the base model, then re-freeze everything except the top 30 layers
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

# Keep Batch Normalization layers frozen for stable fine-tuning
for layer in base_model.layers[-30:]:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

print("Trainable base-model layers:", sum(layer.trainable for layer in base_model.layers))

# Recompile with a very small learning rate so the pretrained weights only change slightly
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

fine_tune_early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

# initial_epoch continues the epoch count from where Phase 1 stopped
phase1_epochs = len(history.history["loss"])

fine_tune_history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=phase1_epochs + 10,
    initial_epoch=phase1_epochs,
    callbacks=[fine_tune_early_stopping]
)

# Phase 2 result: compare with Phase 1
phase2_val_loss, phase2_val_accuracy = model.evaluate(val_ds, verbose=0)

print(f"\nPhase 1 validation accuracy (frozen base): {phase1_val_accuracy:.2%}")
print(f"Phase 2 validation accuracy (fine-tuned):  {phase2_val_accuracy:.2%}")
print(f"Change from fine-tuning: {phase2_val_accuracy - phase1_val_accuracy:+.2%}")

In [ ]:

# Evaluate the trained model on the test dataset
test_loss, test_accuracy = model.evaluate(test_ds, verbose=1)

print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.2%}")


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Step 6: Classification report and confusion matrix
# test_ds has shuffle=False, so the labels stay in the same order as the predictions
y_true = np.concatenate([labels.numpy() for _, labels in test_ds]).astype(int).flatten()
test_probabilities = model.predict(test_ds, verbose=1).flatten()
y_pred = (test_probabilities >= 0.5).astype(int)

# Per-class precision, recall and F1
print(classification_report(y_true, y_pred, target_names=class_names))

# Rows are the actual class, columns are the predicted class
cm = confusion_matrix(y_true, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues")
plt.title("Confusion Matrix - Test Set")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Select three cats and two dogs from the test set
test_batches = list(test_ds)

cat_images, cat_labels = test_batches[0]
dog_images, dog_labels = test_batches[-1]

images = tf.concat([cat_images[:3], dog_images[:2]], axis=0)
labels = tf.concat([cat_labels[:3], dog_labels[:2]], axis=0)

probabilities = model.predict(images, verbose=0).flatten()

plt.figure(figsize=(15, 5))

for i in range(5):
    dog_probability = float(probabilities[i])
    predicted_index = 1 if dog_probability >= 0.5 else 0
    confidence = (
        dog_probability if predicted_index == 1
        else 1 - dog_probability
    )
    true_index = int(labels[i].numpy()[0])

    plt.subplot(1, 5, i + 1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(
        f"Predicted: {class_names[predicted_index]}\n"
        f"Confidence: {confidence:.1%}\n"
        f"Actual: {class_names[true_index]}"
    )
    plt.axis("off")

plt.tight_layout()
plt.show()